# Projeto 2 | Busca semântica em anúncios com CLIP

No pré-treinamento contrastivo, CLIP aproxima representações de imagens e textos correspondentes e afasta as de pares não correspondentes. Seus dois encoders usam o mesmo espaço de embeddings. Assim, podemos comparar anúncios com novas descrições sem treinar um classificador neste corpus.

Usamos o ADS-16 para contar imagens próximas de 20 conceitos e buscar anúncios para oito consultas textuais. Esses scores medem similaridade, não probabilidade calibrada ou presença comprovada de um objeto.

A fonte pública reúne 20 categorias com 15 anúncios cada.

Fonte: [ADS-16 no Kaggle](https://www.kaggle.com/datasets/groffo/ads16-dataset). A atribuição e os detalhes do benchmark estão no README. Usamos o modelo em avaliação, sem gradientes, treino ou fine-tuning.

## Requisitos e planejamento da execução local

| Recurso | Requisito / estimativa |
|---|---|
| RAM disponível recomendada | 8 GB |
| VRAM para a configuração padrão | 2 GB (6 GB recomendados) |
| Tempo estimado com dados e pesos em cache | 2–10 minutos |
| Hardware de referência | NVIDIA GeForce RTX 3050 Laptop, 6 GB de VRAM, Windows, Python 3.12 |
| Protocolo considerado | inferência de 300 anúncios, batch 32; CPU: reservar 2–10 minutos com os arquivos em cache |

Os valores são estimativas de planejamento, não medições de pico nem garantias de tempo. Feche aplicações que ocupem a GPU. Download inicial, armazenamento, outras cargas e early stopping alteram a duração. Instale as dependências de `requirements.txt` na pasta desta atividade e selecione o kernel local. Execute a partir dessa pasta; não há requisito de T4/Colab.


In [1]:
CONTRATO_PROJETO = {
    "dataset": "ADS-16",
    "expected_images": 300,
    "expected_categories": 20,
    "images_per_category": 15,
    "training": False,
    "tasks": ["semantic_frequency", "text_to_image_search"],
}

assert CONTRATO_PROJETO["expected_images"] == 300
assert CONTRATO_PROJETO["expected_categories"] == 20
assert CONTRATO_PROJETO["training"] is False


## 1. Ambiente de inferência

Fixamos a semente para as operações reproduzíveis. Usamos CUDA quando disponível e CPU como alternativa explícita. O batch inicial fica em uma única constante; se faltar memória, sua redução deve ser registrada.

In [2]:
import hashlib
import random
from pathlib import Path

import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image

SEMENTE_ALEATORIA = 42
random.seed(SEMENTE_ALEATORIA)
np.random.seed(SEMENTE_ALEATORIA)
torch.manual_seed(SEMENTE_ALEATORIA)
torch.cuda.manual_seed_all(SEMENTE_ALEATORIA)

DISPOSITIVO = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TAMANHO_LOTE = 32
registro_ambiente = {
    "device": str(DISPOSITIVO),
    "batch_size": TAMANHO_LOTE,
    "torch": torch.__version__,
    "cuda_runtime": torch.version.cuda,
}
if DISPOSITIVO.type == "cuda":
    registro_ambiente["gpu"] = torch.cuda.get_device_name(0)
print(registro_ambiente)


{'device': 'cpu', 'batch_size': 32, 'torch': '2.9.0+cu130', 'cuda_runtime': '13.0'}


## 2. Download e localização dos anúncios

Baixamos o pacote pelo identificador oficial do KaggleHub. Como a árvore `Corpus` também contém imagens coletadas de participantes, selecionamos somente raízes `Ads` com categorias numéricas como filhos imediatos. A busca não depende de caminhos da máquina.

Na versão 1, há um arquivo excedente conhecido: `categoria 1 / arquivo 16.png`. A exclusão autorizada exige que exatamente um arquivo ocupe esse slot. Se o pacote mudar, a checagem interrompe a execução.

Reutilizamos a versão 1 completa no cache oficial, quando disponível. O caminho vem da configuração do KaggleHub; se não houver cache, baixamos pela mesma API e identificador. A validação de 300 anúncios e a exclusão auditada do slot excedente continuam obrigatórias.

In [3]:
EXTENSOES_IMAGENS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
from kagglehub.cache import Cache
from kagglehub.handle import parse_dataset_handle

versao_referencia = parse_dataset_handle("groffo/ads16-dataset").with_version(1)
caminho_em_cache = Cache().load_from_cache(versao_referencia)
raiz_dataset = Path(caminho_em_cache if caminho_em_cache else kagglehub.dataset_download("groffo/ads16-dataset"))
print("Origem:", "cache oficial KaggleHub, versão 1" if caminho_em_cache else "download oficial KaggleHub")

raizes_anuncios = sorted({
    caminho.parent.parent
    for caminho in raiz_dataset.rglob("*")
    if caminho.is_file()
    and caminho.suffix.lower() in EXTENSOES_IMAGENS
    and caminho.parent.name.isdigit()
    and caminho.parent.parent.name == "Ads"
})

caminhos_imagens_candidatas = sorted(
    (caminho for raiz_anuncios in raizes_anuncios for caminho in raiz_anuncios.glob("*/*")
     if caminho.is_file() and caminho.suffix.lower() in EXTENSOES_IMAGENS
     and not any(parte.startswith(".") for parte in caminho.relative_to(raiz_dataset).parts)),
    key=lambda caminho: caminho.relative_to(raiz_dataset).as_posix(),
)

if not caminhos_imagens_candidatas:
    raise FileNotFoundError(f"Nenhum anúncio encontrado em {raiz_dataset}")

POSICAO_EXCEDENTE = ("1", "16")
caminhos_excluidos = [
    caminho for caminho in caminhos_imagens_candidatas
    if (caminho.parent.name, caminho.stem) == POSICAO_EXCEDENTE
]
if len(caminhos_excluidos) != 1:
    raise RuntimeError(
        "A exceção documentada esperava exatamente um arquivo no slot 1/16, "
        f"mas encontrou {len(caminhos_excluidos)}. Nenhum arquivo foi selecionado."
    )
caminhos_imagens = [caminho for caminho in caminhos_imagens_candidatas if caminho not in caminhos_excluidos]

print(f"Raiz: {raiz_dataset}")
print(f"Raízes de anúncios: {len(raizes_anuncios)}")
print(f"Imagens candidatas: {len(caminhos_imagens_candidatas)}")
print(f"Exclusão não canônica: {caminhos_excluidos[0].relative_to(raiz_dataset).as_posix()}")
print(f"Imagens canônicas: {len(caminhos_imagens)}")


Origem: cache oficial KaggleHub, versão 1
Raiz: C:\Users\lord_\.cache\kagglehub\datasets\groffo\ads16-dataset\versions\1
Raízes de anúncios: 2
Imagens candidatas: 301
Exclusão não canônica: ADS16_Benchmark_part1/ADS16_Benchmark_part1/Ads/Ads/1/16.png
Imagens canônicas: 300


## 3. Catálogo e validação do contrato

Abrimos e verificamos cada arquivo, registrando suas informações no catálogo. SHA-256 detecta duplicatas exatas, e o diretório do dataset fornece a categoria. Antes de carregar o CLIP, conferimos quantidade, unicidade e distribuição. Qualquer divergência interrompe a execução e informa os valores encontrados.

In [4]:
registros = []
for caminho in caminhos_imagens:
    with Image.open(caminho) as imagem:
        imagem.verify()
    with Image.open(caminho) as imagem:
        largura, altura = imagem.size
        modo = imagem.mode
    registros.append({
        "path": caminho,
        "relative_path": caminho.relative_to(raiz_dataset).as_posix(),
        "category": caminho.parent.name,
        "width": largura,
        "height": altura,
        "mode": modo,
        "sha256": hashlib.sha256(caminho.read_bytes()).hexdigest(),
    })

dados_imagens = pd.DataFrame(registros).sort_values("relative_path").reset_index(drop=True)
contagens_categorias = dados_imagens["category"].value_counts().sort_index(key=lambda valores: valores.astype(int))
posicoes_esperadas = {(str(categoria), str(indice)) for categoria in range(1, 21) for indice in range(1, 16)}
posicoes_observadas = {(caminho.parent.name, caminho.stem) for caminho in caminhos_imagens}
auditoria = {
    "source_candidates": len(caminhos_imagens_candidatas),
    "excluded_noncanonical": [caminho.relative_to(raiz_dataset).as_posix() for caminho in caminhos_excluidos],
    "files": len(dados_imagens),
    "unique_hashes": dados_imagens["sha256"].nunique(),
    "categories": len(contagens_categorias),
    "counts_by_category": contagens_categorias.to_dict(),
    "missing_expected_slots": sorted(posicoes_esperadas - posicoes_observadas),
    "unexpected_slots": sorted(posicoes_observadas - posicoes_esperadas),
}
print(auditoria)

contrato_valido = (
    len(dados_imagens) == CONTRATO_PROJETO["expected_images"]
    and dados_imagens["sha256"].nunique() == CONTRATO_PROJETO["expected_images"]
    and len(contagens_categorias) == CONTRATO_PROJETO["expected_categories"]
    and contagens_categorias.eq(CONTRATO_PROJETO["images_per_category"]).all()
    and not auditoria["missing_expected_slots"]
    and not auditoria["unexpected_slots"]
)

if not contrato_valido:
    raise RuntimeError(
        "A versão obtida do ADS-16 não satisfaz o contrato de 300 imagens únicas, "
        "20 categorias, 15 anúncios por categoria e slots canônicos completos. "
        "A execução foi interrompida "
        f"sem excluir ou duplicar arquivos. Auditoria observada: {auditoria}"
    )


{'source_candidates': 301, 'excluded_noncanonical': ['ADS16_Benchmark_part1/ADS16_Benchmark_part1/Ads/Ads/1/16.png'], 'files': 300, 'unique_hashes': 300, 'categories': 20, 'counts_by_category': {'1': 15, '2': 15, '3': 15, '4': 15, '5': 15, '6': 15, '7': 15, '8': 15, '9': 15, '10': 15, '11': 15, '12': 15, '13': 15, '14': 15, '15': 15, '16': 15, '17': 15, '18': 15, '19': 15, '20': 15}, 'missing_expected_slots': [], 'unexpected_slots': []}


### Amostra dos anúncios

Exibimos 12 exemplos escolhidos com semente fixa para examinar o corpus. A galeria não interfere no ranking nem nas consultas.

In [5]:
dados_amostra = dados_imagens.sample(n=12, random_state=SEMENTE_ALEATORIA)
figura, eixos = plt.subplots(3, 4, figsize=(14, 10))
for eixo, registro in zip(eixos.flat, dados_amostra.itertuples()):
    with Image.open(registro.path) as imagem:
        eixo.imshow(imagem.convert("RGB"))
    eixo.set_title(f"Categoria {registro.category} | {Path(registro.relative_path).name}")
    eixo.axis("off")
plt.tight_layout()


<Figure size 1400x1000 with 12 Axes>

## Descrições, consultas e threshold

As 20 descrições em inglês incluem produtos, pessoas, veículos, cenas e sinais publicitários. Mantemos os textos do protocolo anterior. As oito consultas incluem objetos específicos, cenas e associações abstratas.

O threshold determina quais scores entram na contagem de ocorrências semânticas. Comparamos 0,20, 0,25 e 0,30 e usamos 0,25 no ranking. Esse corte é uma escolha operacional para o corpus, sem validação como detector de objetos.

### Tokenização do CLIP e do BERT

CLIP usa BPE; BERT usa WordPiece. Ambos dividem o texto em tokens, acrescentam tokens especiais e podem usar padding para agrupar sequências de tamanhos diferentes. A attention mask indica as posições válidas para que o padding não entre como conteúdo.

O encoder textual do CLIP usa também uma máscara causal interna e obtém o embedding associado ao final do texto. BERT representa os tokens com contexto bidirecional. Neste notebook, o processor prepara os tensores e a attention mask; não construímos as máscaras manualmente.

In [6]:
DESCRICOES_CONCEITOS = {
    "person": "an advertisement featuring a person",
    "vehicle": "an advertisement featuring a car or another vehicle",
    "food": "an advertisement featuring food",
    "beverage": "an advertisement featuring a beverage",
    "electronics": "an advertisement featuring an electronic device",
    "clothing": "an advertisement featuring clothing or shoes",
    "beauty": "an advertisement featuring a beauty or cosmetic product",
    "child": "an advertisement featuring a baby or child",
    "animal": "an advertisement featuring an animal or pet",
    "sports": "an advertisement featuring sports or exercise equipment",
    "jewelry": "an advertisement featuring jewelry or a watch",
    "instrument": "an advertisement featuring a musical instrument",
    "tools": "an advertisement featuring tools or hardware",
    "home": "an advertisement featuring furniture or a home interior",
    "office": "an advertisement featuring stationery or office supplies",
    "nature": "an advertisement featuring outdoor scenery or nature",
    "package": "an advertisement featuring product packaging",
    "isolated_product": "an advertisement showing a single product on a plain background",
    "text_logo": "an advertisement dominated by text and a logo",
    "people_using_product": "an advertisement showing people using a product",
}

CONSULTAS_BUSCA = [
    "an advertisement for a consumer product",
    "a product on a plain background",
    "a person using a product",
    "a car driving outdoors",
    "a healthy food product",
    "a luxury watch with an elegant design",
    "a colorful advertisement made for children",
    "an advertisement that suggests excitement and adventure",
]
THRESHOLDS_CANDIDATOS = (0.20, 0.25, 0.30)

assert len(DESCRICOES_CONCEITOS) == len(set(DESCRICOES_CONCEITOS.values())) == 20
assert len(CONSULTAS_BUSCA) == len(set(CONSULTAS_BUSCA)) == 8


## 5. Carregamento e smoke test do CLIP

Depois de validar o corpus, carregamos o modelo e o processador do mesmo checkpoint. Convertemos a imagem para RGB em memória e testamos a passagem em modo de avaliação, dentro de `torch.inference_mode()`.

In [7]:
from transformers import AutoProcessor, CLIPModel

IDENTIFICADOR_MODELO = "openai/clip-vit-base-patch32"
processador = AutoProcessor.from_pretrained(IDENTIFICADOR_MODELO)
modelo = CLIPModel.from_pretrained(IDENTIFICADOR_MODELO).to(DISPOSITIVO).eval()

with Image.open(dados_imagens.loc[0, "path"]) as imagem:
    imagem_amostra = imagem.convert("RGB")

entradas = processador(
    text=[DESCRICOES_CONCEITOS["person"]],
    images=[imagem_amostra],
    return_tensors="pt",
    padding=True,
)
entradas = {nome: tensor.to(DISPOSITIVO) for nome, tensor in entradas.items()}

with torch.inference_mode():
    saidas = modelo(**entradas)

assert saidas.image_embeds.shape[0] == 1
assert saidas.text_embeds.shape[0] == 1
assert saidas.image_embeds.shape[1] == saidas.text_embeds.shape[1]
assert not modelo.training
print(saidas.image_embeds.shape, saidas.text_embeds.shape, DISPOSITIVO)


torch.Size([1, 512]) torch.Size([1, 512]) cpu


Loading weights: 100%|##########| 398/398 [00:00<00:00, 39589.55it/s]


## 6. Embeddings normalizados das imagens

Codificamos as 300 imagens na ordem do catálogo, sem embaralhar. Cada lote é convertido para RGB em memória, processado em inferência e normalizado pela norma L2. Transferimos os lotes concluídos para CPU para liberar VRAM. Cada linha da matriz continua correspondendo à mesma entrada de `dados_imagens`.

In [8]:
import torch.nn.functional as F
from tqdm.auto import trange

lotes_embeddings_imagens = []
caminhos_processados = []

for inicio in trange(0, len(dados_imagens), TAMANHO_LOTE, desc="Imagens"):
    caminhos_lote = dados_imagens.iloc[inicio:inicio + TAMANHO_LOTE]["path"].tolist()
    imagens_lote = []
    for caminho in caminhos_lote:
        with Image.open(caminho) as imagem:
            imagens_lote.append(imagem.convert("RGB"))

    pixel_values = processador(
        images=imagens_lote,
        return_tensors="pt",
    )["pixel_values"].to(DISPOSITIVO)

    with torch.inference_mode():
        saida_caracteristicas_imagens = modelo.get_image_features(pixel_values=pixel_values)
        embeddings_lote = (
            saida_caracteristicas_imagens.pooler_output
            if hasattr(saida_caracteristicas_imagens, "pooler_output")
            else saida_caracteristicas_imagens
        )
        embeddings_lote = F.normalize(embeddings_lote, dim=-1)

    lotes_embeddings_imagens.append(embeddings_lote.cpu())
    caminhos_processados.extend(caminhos_lote)

embeddings_imagens = torch.cat(lotes_embeddings_imagens, dim=0)
normas_embeddings_imagens = embeddings_imagens.norm(dim=-1)
caminhos_catalogo = dados_imagens["path"].tolist()

assert caminhos_processados == caminhos_catalogo
assert embeddings_imagens.shape == (CONTRATO_PROJETO["expected_images"], saidas.image_embeds.shape[1])
assert embeddings_imagens.device.type == "cpu"
assert not embeddings_imagens.requires_grad
assert torch.isfinite(embeddings_imagens).all()
assert torch.allclose(
    normas_embeddings_imagens,
    torch.ones(CONTRATO_PROJETO["expected_images"]),
    atol=1e-5,
)
assert not modelo.training

print("Embeddings de imagens:", tuple(embeddings_imagens.shape), "| normas unitárias verificadas.")


Embeddings de imagens: (300, 512) | normas unitárias verificadas.


Imagens: 100%|##########| 10/10 [00:30<00:00,  3.07s/it]


### Organizar as figuras

Salvamos as figuras desta versão em resultados/v3 e preservamos as anteriores. Os títulos usam caminhos curtos para facilitar a leitura.

In [9]:
CAMINHO_FIGURAS_BUSCA = Path("resultados/v3/figuras/consultas")
CAMINHO_FIGURAS_CONCEITOS = Path("resultados/v3/figuras/conceitos")

def resumir_caminho(caminho_relativo):
    partes = Path(caminho_relativo).parts
    return Path(*partes[-3:]).as_posix()

CAMINHO_FIGURAS_BUSCA.mkdir(parents=True, exist_ok=True)
CAMINHO_FIGURAS_CONCEITOS.mkdir(parents=True, exist_ok=True)
print(f"Diretório de figuras de busca: {CAMINHO_FIGURAS_BUSCA}")
print(f"Diretório de figuras dos conceitos líderes: {CAMINHO_FIGURAS_CONCEITOS}")


Diretório de figuras de busca: resultados\v3\figuras\consultas
Diretório de figuras dos conceitos líderes: resultados\v3\figuras\conceitos


## 7. Ranking de conceitos por frequência semântica

Codificamos os 20 conceitos fixados na etapa 4 com o mesmo checkpoint das imagens. O produto matricial dos embeddings normalizados gera a matriz de similaridade `300 × 20`, na ordem do catálogo.

Comparamos os thresholds `0,20`, `0,25` e `0,30` em uma tabela de sensibilidade. Para quatro conceitos (texto/logotipo, produto isolado, pessoa e veículo), examinamos os scores logo abaixo e acima dos cortes. Caminho e categoria ajudam a conferir os exemplos. O corte definitivo é fixado antes do ranking final, com base nessa inspeção, e continua sendo um limite de similaridade, não uma probabilidade.

In [10]:
def extrair_embeddings_texto(textos):
    entradas = processador(text=textos, return_tensors="pt", padding=True)
    entradas = {nome: tensor.to(DISPOSITIVO) for nome, tensor in entradas.items()}
    with torch.inference_mode():
        saida = modelo.get_text_features(**entradas)
        embeddings = saida.pooler_output if hasattr(saida, "pooler_output") else saida
        return F.normalize(embeddings, dim=-1).cpu()


### Similaridade e sensibilidade ao corte

Comparamos imagens e conceitos no mesmo espaço de embeddings. Antes de aplicar o corte fixado, mostramos como a contagem varia nos três thresholds.

In [11]:
nomes_conceitos = list(DESCRICOES_CONCEITOS)
textos_conceitos = list(DESCRICOES_CONCEITOS.values())

embeddings_conceitos = extrair_embeddings_texto(textos_conceitos)

assert embeddings_conceitos.shape == (len(DESCRICOES_CONCEITOS), embeddings_imagens.shape[1])
assert torch.isfinite(embeddings_conceitos).all()
assert not embeddings_conceitos.requires_grad
assert torch.allclose(
    embeddings_conceitos.norm(dim=-1),
    torch.ones(len(DESCRICOES_CONCEITOS)),
    atol=1e-5,
)

similaridades = embeddings_imagens @ embeddings_conceitos.T
assert similaridades.shape == (CONTRATO_PROJETO["expected_images"], len(DESCRICOES_CONCEITOS))
assert torch.isfinite(similaridades).all()

registros_sensibilidade = []
for threshold in THRESHOLDS_CANDIDATOS:
    contagens = (similaridades >= threshold).sum(dim=0)
    for conceito, quantidade in zip(nomes_conceitos, contagens.tolist()):
        registros_sensibilidade.append({
            "threshold": threshold,
            "concept": conceito,
            "occurrences": int(quantidade),
            "occurrence_rate": float(quantidade) / CONTRATO_PROJETO["expected_images"],
        })
dados_sensibilidade = pd.DataFrame(registros_sensibilidade)

display(dados_sensibilidade)


    threshold               concept  occurrences  occurrence_rate
0        0.20                person          245         0.816667
1        0.20               vehicle          178         0.593333
2        0.20                  food          182         0.606667
3        0.20              beverage          163         0.543333
4        0.20           electronics          226         0.753333
5        0.20              clothing          177         0.590000
6        0.20                beauty          188         0.626667
7        0.20                 child          195         0.650000
8        0.20                animal          198         0.660000
9        0.20                sports          192         0.640000
10       0.20               jewelry          137         0.456667
11       0.20            instrument          183         0.610000
12       0.20                 tools          231         0.770000
13       0.20                  home          155         0.516667
14       0

### Exemplos próximos dos thresholds

Examinamos imagens dos quatro conceitos com scores logo abaixo e acima de cada corte. Esses exemplos permitem conferir o efeito da escolha.

In [12]:
conceitos_inspecionar = ["text_logo", "isolated_product", "person", "vehicle"]
registros_inspecao = []

for conceito in conceitos_inspecionar:
    indice_conceito = nomes_conceitos.index(conceito)
    pontuacoes = similaridades[:, indice_conceito]

    for threshold in THRESHOLDS_CANDIDATOS:
        for lado, mascara, selecionar in (
            ("abaixo", pontuacoes < threshold, torch.argmax),
            ("acima_ou_igual", pontuacoes >= threshold, torch.argmin),
        ):
            indices_candidatos = torch.where(mascara)[0]
            if indices_candidatos.numel() == 0:
                registros_inspecao.append({
                    "concept": conceito, "threshold": threshold, "side": lado,
                    "image_index": None, "score": float("nan"),
                    "relative_path": "", "category": "",
                })
                continue

            pontuacoes_candidatas = pontuacoes[indices_candidatos]
            indice_imagem = indices_candidatos[selecionar(pontuacoes_candidatas)].item()
            registro = dados_imagens.iloc[indice_imagem]
            registros_inspecao.append({
                "concept": conceito,
                "threshold": threshold,
                "side": lado,
                "image_index": indice_imagem,
                "score": float(pontuacoes[indice_imagem]),
                "relative_path": registro["relative_path"],
                "category": registro["category"],
            })

dados_inspecao = pd.DataFrame(registros_inspecao)
for registro in registros_inspecao:
    if registro["image_index"] is not None:
        indice_conceito = nomes_conceitos.index(registro["concept"])
        assert np.isclose(registro["score"], similaridades[registro["image_index"], indice_conceito].item())
display(dados_inspecao)


             concept  ...  category
0          text_logo  ...         3
1          text_logo  ...         3
2          text_logo  ...         8
3          text_logo  ...        11
4          text_logo  ...        17
5          text_logo  ...        10
6   isolated_product  ...         5
7   isolated_product  ...        17
8   isolated_product  ...        17
9   isolated_product  ...         7
10  isolated_product  ...        20
11  isolated_product  ...        16
12            person  ...         7
13            person  ...         7
14            person  ...        12
15            person  ...         1
16            person  ...         2
17            person  ...        17
18           vehicle  ...         3
19           vehicle  ...         3
20           vehicle  ...        12
21           vehicle  ...        16
22           vehicle  ...         2
23           vehicle  ...         2

[24 rows x 7 columns]

### 7.1 Manter o threshold 0,25

Com 0,20, muitas imagens entram por correspondências amplas; com 0,30, poucas ficam acima do limite. Mantemos 0,25, previsto no protocolo, e usamos os demais cortes na análise de sensibilidade. Os exemplos próximos dos limites ajudam a examinar os casos de fronteira.

A contagem expressa compatibilidade com a descrição, sem validação manual dos objetos. As categorias não alteram os scores ou o ranking. O corte, portanto, não garante precisão semântica.

In [13]:
THRESHOLD_SELECIONADO = 0.25
assert THRESHOLD_SELECIONADO in THRESHOLDS_CANDIDATOS

ranking_conceitos = pd.DataFrame({
    "concept": nomes_conceitos,
    "mean_similarity": similaridades.mean(dim=0).numpy(),
    "occurrences": (similaridades >= THRESHOLD_SELECIONADO).sum(dim=0).numpy(),
})
ranking_conceitos["occurrence_rate"] = ranking_conceitos["occurrences"] / CONTRATO_PROJETO["expected_images"]
ranking_conceitos = ranking_conceitos.sort_values(
    by=["occurrences", "mean_similarity", "concept"],
    ascending=[False, False, True],
).reset_index(drop=True)

assert (ranking_conceitos["occurrences"] <= CONTRATO_PROJETO["expected_images"]).all()
assert ((ranking_conceitos["occurrence_rate"] >= 0) & (ranking_conceitos["occurrence_rate"] <= 1)).all()
display(ranking_conceitos)


                 concept  mean_similarity  occurrences  occurrence_rate
0       isolated_product         0.258011          197         0.656667
1   people_using_product         0.252479          164         0.546667
2              text_logo         0.238946          114         0.380000
3                package         0.236065           75         0.250000
4                 person         0.226496           56         0.186667
5                 office         0.220236           42         0.140000
6                  tools         0.219092           35         0.116667
7                 nature         0.219620           34         0.113333
8            electronics         0.219727           31         0.103333
9                  child         0.213285           31         0.103333
10              clothing         0.206640           25         0.083333
11               vehicle         0.204014           23         0.076667
12            instrument         0.208823           20         0

## 8. Visualizar os cinco conceitos líderes

Para cada conceito líder, mostramos as três imagens de maior score, sem escolha manual. Uma imagem pode aparecer em mais de um conceito. A figura fica em `resultados/v3/figuras/conceitos/top_concepts_5x3.png`.

- Produto isolado: aparecem dois anúncios de hambúrguer e uma peça textual. Nem todos mostram um único produto em fundo neutro.
- Pessoas usando produto: os resultados incluem uma pessoa no Badoo, a interface Lovoo e um anúncio textual. Eles não confirmam uma ação de uso.
- Texto/logotipo: slogans e marcas grandes aparecem nas peças de Burger King, Fiat e McDonald's, com correspondência visual clara.
- Embalagem: os anúncios de produtos infantis, antivírus e creme dental mostram recipientes ou caixas, junto a pessoas e texto.
- Pessoa: Badoo e Kaspersky mostram figuras humanas; o anúncio de carro é um falso positivo para a presença de pessoa.

Esses resultados mostram o limite da contagem: proximidade semântica não equivale a contar objetos visíveis.

### Confirmação visual dos exemplos da execução salva

Inspecionamos a grade com seus caminhos; esta conferência não altera o ranking nem as frequências. A interpretação se restringe aos quinze exemplos exibidos, sem alegar anotação integral do corpus.

| Conceito líder | Exemplo e confirmação visual | Limite |
|---|---|---|
| Produto isolado em fundo liso | `Ads/10/15.png` mostra três hambúrgueres; `Ads/10/11.png` mostra hambúrguer e copo | Nenhum dos três exemplos líderes confirma **um único produto**. `Ads/3/3.png` é textual. Esta galeria não oferece confirmação do conceito completo. |
| Pessoas usando produto | `Ads/20/15.png` mostra mãos segurando um celular com anúncio do Lovoo | Evidência parcial de interação com o aparelho; não comprova uso do serviço anunciado. `Ads/20/11.png` mostra uma pessoa posando, e `Ads/3/3.png` contém texto. Não há confirmação inequívoca da ação completa nos três líderes. |
| Texto e logotipo | `Ads/10/11.png`, `Ads/2/11.png`, `Ads/10/15.png`: slogans e marcas visíveis | Confirma presença de texto e marcas; a descrição “dominado por texto” continua subjetiva. |
| Embalagem | `Ads/3/11.png`, `Ads/17/11.png`, `Ads/4/14.png`: recipientes e caixas de produtos visíveis | A presença da embalagem é confirmada; não demonstra uso ou compra. |
| Pessoa | `Ads/20/11.png` (homem) e `Ads/17/11.png` (homem com caixas) | Dois exemplos confirmados. `Ads/2/11.png` mostra carro/logotipo e é falso positivo para pessoa. |

Para os conceitos sem confirmação inequívoca, registramos a insuficiência dos exemplos recuperados, em vez de chamar similaridade de presença. Não concluímos que o conceito inexiste nas outras imagens do corpus, que não receberam anotação de presença. A revisão visual é interpretativa e não uma validação humana especializada.


A execução v3 foi concluída em CPU local com dois threads, em 08/10/2026. Os líderes permaneceram os mesmos. A contagem de `people_using_product` foi 164/300, contra 163/300 na execução CUDA v2: diferenças numéricas pequenas podem deslocar uma imagem próxima do corte. Não alteramos prompts, corpus ou threshold. Os caminhos e a ordem das oito buscas permaneceram iguais. A galeria complementar seguinte confirma um exemplo positivo para cada conceito, preservando a análise dos falsos positivos do top-3.

In [14]:
conceitos_lideres = ranking_conceitos.head(5)["concept"].tolist()
figura, eixos = plt.subplots(len(conceitos_lideres), 3, figsize=(13, 19))

for indice_linha, conceito in enumerate(conceitos_lideres):
    indice_conceito = nomes_conceitos.index(conceito)
    pontuacoes = similaridades[:, indice_conceito]
    pontuacoes_lideres, indices_lideres = pontuacoes.topk(3)

    for coluna, (pontuacao, indice_imagem) in enumerate(zip(pontuacoes_lideres, indices_lideres)):
        registro = dados_imagens.iloc[indice_imagem.item()]
        with Image.open(registro["path"]) as imagem:
            eixos[indice_linha, coluna].imshow(imagem.convert("RGB"))
        eixos[indice_linha, coluna].set_title(
            f"{conceito} #{coluna + 1}\n"
            f"score={pontuacao.item():.3f}\n{resumir_caminho(registro['relative_path'])}\n"
            f"categoria={registro['category']}",
            fontsize=8,
        )
        eixos[indice_linha, coluna].axis("off")

figura.suptitle("Cinco conceitos mais frequentes (top-3 por similaridade)", fontsize=14)
plt.tight_layout(rect=(0, 0, 1, 0.96))
figura.savefig(CAMINHO_FIGURAS_CONCEITOS / "top_concepts_5x3.png", dpi=110, bbox_inches="tight")
plt.show()

print(f"Figura salva em {CAMINHO_FIGURAS_CONCEITOS / 'top_concepts_5x3.png'}")
print("Conceitos líderes:", conceitos_lideres)


Figura salva em resultados\v3\figuras\conceitos\top_concepts_5x3.png
Conceitos líderes: ['isolated_product', 'people_using_product', 'text_logo', 'package', 'person']


<Figure size 1300x1900 with 15 Axes>

### Galeria complementar com presença visual confirmada

Preservamos o top-3 automático e seus falsos positivos. Depois de inspecionar anúncios do corpus, escolhemos um exemplo positivo por conceito líder. A seleção visual foi feita antes de consultar seus scores e não recalcula ranking, frequências ou threshold. O relógio `13/7.png` está sozinho em fundo branco; o jogador de `18/12.png` usa os calçados em uma ação esportiva. Os outros exemplos mostram marca/texto, embalagens e uma pessoa.

O score original é mostrado para conferir se o exemplo contribuiu à frequência acima do corte. Mesmo um exemplo visualmente positivo pode ficar abaixo de 0,25; isso é uma limitação do detector semântico. A inspeção é assistida, sem anotação integral de presença no corpus nem estimativa de precisão/recall de objetos. As observações identificam elementos concretos e não certificam atributos subjetivos ou uso efetivo de serviços.


In [15]:
EXEMPLOS_CONFIRMADOS = {
    'isolated_product': ('13', '7', 'Um relógio completo é mostrado sozinho em fundo branco; o anúncio traz texto abaixo.'),
    'people_using_product': ('18', '12', 'Um jogador chuta uma bola usando calçados esportivos; o produto aparece em uso.'),
    'text_logo': ('2', '11', 'O logotipo Fiat e os blocos de texto ocupam parte expressiva da peça.'),
    'package': ('4', '14', 'Recipientes e caixas Colgate aparecem junto da modelo.'),
    'person': ('20', '11', 'Um homem aparece em primeiro plano no anúncio Badoo.'),
}
assert set(EXEMPLOS_CONFIRMADOS) == set(conceitos_lideres)
registros_confirmacao = []
figura_confirmacao, eixos_confirmacao = plt.subplots(1, 5, figsize=(17, 6))
for eixo, conceito in zip(eixos_confirmacao, conceitos_lideres):
    categoria, indice, observacao = EXEMPLOS_CONFIRMADOS[conceito]
    selecionados = dados_imagens.index[
        dados_imagens['path'].map(lambda caminho: caminho.parent.name == categoria and caminho.stem == indice)
    ].tolist()
    assert len(selecionados) == 1, f'Exemplo ausente ou ambíguo: {categoria}/{indice}'
    indice_imagem = selecionados[0]
    registro = dados_imagens.iloc[indice_imagem]
    score = float(similaridades[indice_imagem, nomes_conceitos.index(conceito)].item())
    with Image.open(registro['path']) as imagem:
        eixo.imshow(imagem.convert('RGB'))
    eixo.set_title(f"{conceito}\n{resumir_caminho(registro['relative_path'])}\nscore={score:.3f}; corte={THRESHOLD_SELECIONADO:.2f}", fontsize=9)
    eixo.axis('off')
    registros_confirmacao.append({'concept': conceito, 'relative_path': registro['relative_path'],
        'sha256': registro['sha256'], 'score': score, 'acima_do_corte': score >= THRESHOLD_SELECIONADO,
        'criterio_visual': observacao, 'origem_selecao': 'Inspecao visual assistida; independente do ranking'})
figura_confirmacao.suptitle('Exemplos positivos inspecionados: complemento ao top-3 automático')
figura_confirmacao.tight_layout()
figura_confirmacao.savefig(CAMINHO_FIGURAS_CONCEITOS / 'exemplos_confirmados.png', dpi=120)
plt.show()
confirmacoes_visuais = pd.DataFrame(registros_confirmacao)
confirmacoes_visuais.to_csv(CAMINHO_FIGURAS_CONCEITOS / 'confirmacoes_visuais.csv', index=False)
display(confirmacoes_visuais[['concept', 'relative_path', 'score', 'acima_do_corte', 'criterio_visual']])


<Figure size 1700x600 with 5 Axes>

                concept  ...                                    criterio_visual
0      isolated_product  ...  Um relógio completo é mostrado sozinho em fund...
1  people_using_product  ...  Um jogador chuta uma bola usando calçados espo...
2             text_logo  ...  O logotipo Fiat e os blocos de texto ocupam pa...
3               package  ...  Recipientes e caixas Colgate aparecem junto da...
4                person  ...  Um homem aparece em primeiro plano no anúncio ...

[5 rows x 5 columns]

## 9. Busca semântica por consulta textual

Codificamos cada consulta com o mesmo checkpoint, normalizamos seu embedding e o comparamos com todas as imagens. A função retorna cinco imagens distintas em ordem decrescente de similaridade. Não usa o threshold do ranking nem as categorias para reordenar os resultados.

Sempre haverá um top-5, mesmo quando os scores forem baixos e a consulta tiver pouca correspondência no corpus.

In [16]:
def buscar_imagens(consulta, quantidade_resultados=5):
    if not isinstance(consulta, str) or not consulta.strip():
        raise ValueError("A consulta deve ser um texto não vazio.")
    if not 1 <= quantidade_resultados <= len(dados_imagens):
        raise ValueError("top_k deve estar entre 1 e o tamanho do corpus.")

    embedding_consulta = extrair_embeddings_texto([consulta.strip()])

    pontuacoes_consulta = embeddings_imagens @ embedding_consulta.T
    pontuacoes_lideres, indices_lideres = pontuacoes_consulta.squeeze(1).topk(quantidade_resultados)

    resultados = dados_imagens.iloc[indices_lideres.tolist()][
        ["path", "relative_path", "category"]
    ].copy()
    resultados.insert(0, "rank", range(1, quantidade_resultados + 1))
    resultados.insert(1, "score", pontuacoes_lideres.numpy())
    return resultados.reset_index(drop=True)

resultados_verificacao = buscar_imagens("a product on a plain background")
assert len(resultados_verificacao) == 5
assert resultados_verificacao["relative_path"].nunique() == 5
assert (resultados_verificacao["score"].to_numpy()[:-1] >= resultados_verificacao["score"].to_numpy()[1:]).all()
display(resultados_verificacao)
print("Smoke test da busca semântica aprovado.")


Smoke test da busca semântica aprovado.


   rank     score  ...                                      relative_path category
0     1  0.267820  ...  ADS16_Benchmark_part1/ADS16_Benchmark_part1/Ad...        3
1     2  0.265204  ...  ADS16_Benchmark_part2/ADS16_Benchmark_part2/Ad...       15
2     3  0.264820  ...  ADS16_Benchmark_part1/ADS16_Benchmark_part1/Ad...        9
3     4  0.263902  ...  ADS16_Benchmark_part2/ADS16_Benchmark_part2/Ad...       13
4     5  0.262694  ...  ADS16_Benchmark_part1/ADS16_Benchmark_part1/Ad...        8

[5 rows x 5 columns]

## 10. Executar as oito consultas

Executamos as consultas já definidas, com descrições genéricas, específicas e abstratas. Cada consulta retorna cinco imagens distintas em ordem de score, com caminho e categoria para conferência. As grades ficam em `resultados/v3/figuras/consultas/`.

Para cada consulta, comparamos a cena esperada com as imagens recuperadas e registramos acertos, ambiguidades e elementos ausentes.

In [17]:
def mostrar_resultados_busca(consulta, resultados, caminho_figura=None):
    figura, eixos = plt.subplots(1, 5, figsize=(18, 4.5))
    for eixo, registro in zip(eixos, resultados.itertuples(index=False)):
        with Image.open(registro.path) as imagem:
            eixo.imshow(imagem.convert("RGB"))
        eixo.set_title(
            f"#{registro.rank} score={registro.score:.3f}\n{resumir_caminho(registro.relative_path)}\ncategoria={registro.category}",
            fontsize=8,
        )
        eixo.axis("off")
    figura.suptitle(consulta, fontsize=12)
    plt.tight_layout()
    if caminho_figura is not None:
        figura.savefig(caminho_figura, dpi=110, bbox_inches="tight")
    plt.show()

resultados_buscas = {}
registros_buscas = []
for indice, consulta in enumerate(CONSULTAS_BUSCA, start=1):
    resultados = buscar_imagens(consulta, quantidade_resultados=5)
    assert len(resultados) == 5
    assert resultados["relative_path"].nunique() == 5
    resultados_buscas[consulta] = resultados
    caminho_figura = CAMINHO_FIGURAS_BUSCA / f"query_{indice:02d}.png"
    mostrar_resultados_busca(consulta, resultados, caminho_figura=caminho_figura)
    for registro in resultados.itertuples(index=False):
        registros_buscas.append({
            "query": consulta,
            "rank": registro.rank,
            "score": float(registro.score),
            "relative_path": registro.relative_path,
            "category": registro.category,
        })

resumo_buscas = pd.DataFrame(registros_buscas)
print(f"Figuras salvas em {CAMINHO_FIGURAS_BUSCA}")
display(resumo_buscas)


Figuras salvas em resultados\v3\figuras\consultas


<Figure size 1800x450 with 5 Axes>

<Figure size 1800x450 with 5 Axes>

<Figure size 1800x450 with 5 Axes>

<Figure size 1800x450 with 5 Axes>

<Figure size 1800x450 with 5 Axes>

<Figure size 1800x450 with 5 Axes>

<Figure size 1800x450 with 5 Axes>

<Figure size 1800x450 with 5 Axes>

                                                query  ...  category
0             an advertisement for a consumer product  ...        20
1             an advertisement for a consumer product  ...         1
2             an advertisement for a consumer product  ...         4
3             an advertisement for a consumer product  ...        17
4             an advertisement for a consumer product  ...         3
5                     a product on a plain background  ...         3
6                     a product on a plain background  ...        15
7                     a product on a plain background  ...         9
8                     a product on a plain background  ...        13
9                     a product on a plain background  ...         8
10                           a person using a product  ...         3
11                           a person using a product  ...         3
12                           a person using a product  ...        18
13                           a per

### 10.1 Leitura das oito consultas

1. Produto de consumo, consulta genérica. A expectativa era recuperar anúncios de varejo. Os resultados misturam aplicativo Badoo, roupas, creme dental, antivírus e produtos infantis, com scores entre 0,290 e 0,304. A descrição ampla trouxe tanto serviços quanto bens físicos, coerente com sua pouca especificidade.

2. Produto em fundo liso, consulta visual específica. A expectativa era encontrar um produto isolado. Três resultados são anúncios textuais em fundo branco; os outros mostram cadeira e garrafa. O resultado atende parcialmente ao pedido: o fundo liso aparece com mais frequência que um produto fotografado.

3. Pessoa usando produto, consulta com ação. Procurávamos uma pessoa em contato com o produto. Aparecem anúncios textuais, um aparelho Scholl e uma mamadeira, sem uma pessoa em uso. A ação pedida não aparece. A proximidade no embedding não garante que alguém esteja usando o produto.

4. Carro em movimento ao ar livre, consulta de cena. Esperávamos um carro em uma pista, com paisagem ao redor. Os resultados são anúncios de autopeças e um cinzeiro para carro, sem a cena externa pedida. Os resultados se relacionam com automóveis, mas não mostram a cena solicitada.

5. Alimento saudável, consulta com atributo. A expectativa era encontrar alimentos com sinais associados à saúde. Aparecem produtos para animais, suplemento de whey e substrato para répteis. Há relação parcial com alimento ou nutrição. Nem todos os exemplos atendem ao pedido, e nenhum deles comprova o atributo saudável.

6. Relógio luxuoso e elegante, consulta estética. Esperávamos relógios com design sofisticado. Quatro resultados mostram relógios e um é anúncio textual de loja, com scores entre 0,226 e 0,259. A busca recupera relógios, mas as imagens não comprovam luxo ou autenticidade das marcas.

7. Anúncio colorido para crianças, consulta de público. Procurávamos cores e elementos associados ao público infantil. Aparecem Kid Robot, McDonald's, Blackboard, Post-it e Blu-ray, vários predominantemente textuais. Alguns resultados atendem ao pedido de cor. A relação com o público infantil, porém, não fica clara, e a descrição completa é pouco atendida.

8. Empolgação e aventura, consulta abstrata. Esperávamos cenas dinâmicas, viagens ou jogos. Tenda de camping e Assassin's Creed oferecem uma associação plausível; anúncio de moda e peças textuais têm relação mais ambígua. O resultado atende parcialmente à consulta e não permite saber se o público percebe a emoção pretendida.

As grades e os caminhos permitem conferir essas observações. Comparamos os scores dentro de cada consulta, sem interpretá-los como probabilidade de presença do conceito.

## 11. Validação final do notebook

Esta célula confere os contratos do catálogo, dos embeddings, do ranking e das buscas e registra o ambiente utilizado. O notebook final executa de cima para baixo em uma sessão limpa, com resultados calculados pelo pipeline.

In [18]:
import platform
import transformers
import torchvision

assert len(dados_imagens) == CONTRATO_PROJETO["expected_images"]
assert dados_imagens["sha256"].nunique() == CONTRATO_PROJETO["expected_images"]
assert dados_imagens["category"].nunique() == CONTRATO_PROJETO["expected_categories"]
assert dados_imagens["category"].value_counts().eq(CONTRATO_PROJETO["images_per_category"]).all()
assert not auditoria["missing_expected_slots"]
assert not auditoria["unexpected_slots"]
assert len(DESCRICOES_CONCEITOS) == 20
assert len(CONSULTAS_BUSCA) == 8
assert embeddings_imagens.shape == (CONTRATO_PROJETO["expected_images"], embeddings_conceitos.shape[1])
assert embeddings_conceitos.shape[0] == 20
assert similaridades.shape == (CONTRATO_PROJETO["expected_images"], 20)
assert torch.allclose(embeddings_imagens.norm(dim=-1), torch.ones(CONTRATO_PROJETO["expected_images"]), atol=1e-5)
assert torch.allclose(embeddings_conceitos.norm(dim=-1), torch.ones(20), atol=1e-5)
assert len(dados_inspecao) == len(conceitos_inspecionar) * len(THRESHOLDS_CANDIDATOS) * 2
assert (dados_inspecao.loc[dados_inspecao["side"] == "abaixo", "score"] < dados_inspecao.loc[dados_inspecao["side"] == "abaixo", "threshold"]).all()
assert (dados_inspecao.loc[dados_inspecao["side"] == "acima_ou_igual", "score"] >= dados_inspecao.loc[dados_inspecao["side"] == "acima_ou_igual", "threshold"]).all()
assert THRESHOLD_SELECIONADO in THRESHOLDS_CANDIDATOS
assert len(ranking_conceitos) == 20
assert len(resultados_buscas) == len(CONSULTAS_BUSCA)
assert all(len(resultados) == 5 for resultados in resultados_buscas.values())
assert all(
    resultados["relative_path"].nunique() == 5
    for resultados in resultados_buscas.values()
)
assert (ranking_conceitos["occurrences"].to_numpy()[:-1] >= ranking_conceitos["occurrences"].to_numpy()[1:]).all()
assert not modelo.training


### Ambiente e contratos

Depois de conferir ranking, consultas e embeddings, registramos o ambiente, as dimensões e a configuração.

In [19]:
print({
    "python": platform.python_version(),
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "transformers": transformers.__version__,
    "model": IDENTIFICADOR_MODELO,
    "device": str(DISPOSITIVO),
    "batch_size": TAMANHO_LOTE,
    "threshold": THRESHOLD_SELECIONADO,
    "image_embeddings_shape": tuple(embeddings_imagens.shape),
    "concept_embeddings_shape": tuple(embeddings_conceitos.shape),
    "similarities_shape": tuple(similaridades.shape),
    "ranking_rows": len(ranking_conceitos),
    "search_queries": len(resultados_buscas),
})


{'python': '3.12.6', 'torch': '2.9.0+cu130', 'torchvision': '0.24.0+cu130', 'transformers': '5.17.0', 'model': 'openai/clip-vit-base-patch32', 'device': 'cpu', 'batch_size': 32, 'threshold': 0.25, 'image_embeddings_shape': (300, 512), 'concept_embeddings_shape': (20, 512), 'similarities_shape': (300, 20), 'ranking_rows': 20, 'search_queries': 8}


### Salvar ranking e consultas

Salvamos as tabelas e o ambiente para comparar novas execuções do protocolo. Embeddings e pesos ficam fora do versionamento.

In [20]:
import json
CAMINHO_RESULTADOS = Path("resultados/v3")
dados_sensibilidade.to_csv(CAMINHO_RESULTADOS / "sensibilidade.csv", index=False)
ranking_conceitos.to_csv(CAMINHO_RESULTADOS / "ranking.csv", index=False)
resumo_buscas.to_csv(CAMINHO_RESULTADOS / "consultas.csv", index=False)
registro_ambiente = {"torch": torch.__version__, "dispositivo": str(DISPOSITIVO), "gpu": torch.cuda.get_device_name(0) if DISPOSITIVO.type == "cuda" else None}
(CAMINHO_RESULTADOS / "ambiente.json").write_text(json.dumps(registro_ambiente, indent=2), encoding="utf-8")


Out[20]: 

67